# Clase 5 · 03 · Comparar y elegir

Con muchos runs, mirar uno por uno no escala. MLflow permite **consultar** los runs como una tabla (`mlflow.search_runs`) y filtrarlos por parámetros, métricas o tags. Este notebook trabaja sobre **tus** runs del notebook 02 (y los del 01): no entrena nada.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/ml_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
T = lambda name: qualified_table(config, name)
client = MlflowClient()

In [ ]:
assert spark.catalog.tableExists(T("ml_features")), "Falta ml_features. Ejecutá primero 00_preflight."
experiment = setup_experiment(spark, config)
data_version = table_version(spark, T("ml_features"))
print("Experimento:", experiment.name)

runs = my_runs(config, experiment.experiment_id)
runs["hiperparametros"] = runs.apply(hyperparams_text, axis=1)
print(f"Runs de modelos: {len(runs)}")
if len(runs) < 5:
    print("⚠️  Tenés menos de 5 runs: volvé al notebook 02 y registrá algunos más.")
test_rate = load_split(spark, config, "test")["chargeback"].mean()
display(runs.drop(columns=["model_uri"]))

## 1. ¿Cuál es el mejor? Depende de la métrica

In [ ]:
show = ["run", "modelo", "features", "class_weight", "umbral", "hiperparametros", "pr_auc", "accuracy", "precision", "recall", "f1"]
best_pr = runs.sort_values("pr_auc", ascending=False).head(1)
best_acc = runs.sort_values("accuracy", ascending=False).head(1)
display(pd.concat([best_pr.assign(criterio="mejor PR-AUC"), best_acc.assign(criterio="mejor accuracy")])[["criterio"] + show])
print(f"Referencia: un modelo al azar tiene PR-AUC ≈ {test_rate:.3f}; decir siempre 'no' da accuracy {1 - test_rate:.3f}.")

Si el mejor por accuracy es un run que casi no alerta, ahí está la trampa: con pocos contracargos, no alertar nunca "acierta" casi siempre. Elegimos por **PR-AUC**, y después ajustamos el **umbral** según cuántas alertas puede revisar el equipo de riesgo.

## 2. Los runs en gráficos

In [ ]:
colors = {"logistic_regression": "#2563eb", "random_forest": "#16a34a", "gradient_boosting": "#f59e0b"}
ordered = runs.sort_values("pr_auc")
fig, axes = plt.subplots(1, 3, figsize=(16, max(3.5, 0.32 * len(runs) + 1.5)))

axes[0].barh(ordered["run"], ordered["pr_auc"], color=[colors.get(m, "#94a3b8") for m in ordered["modelo"]])
axes[0].axvline(test_rate, color="black", ls="--", label="azar")
axes[0].set_xlabel("PR-AUC en test")
axes[0].set_title("Todos tus runs")
axes[0].legend()

for model, group in runs.groupby("modelo"):
    axes[1].scatter(group["recall"], group["precision"], s=60, color=colors.get(model, "#94a3b8"), label=model)
    for row in group.itertuples():
        axes[1].annotate(f"u{row.umbral}", (row.recall, row.precision), xytext=(4, 3), textcoords="offset points", fontsize=7)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision vs. recall (según umbral)")
axes[1].legend(fontsize=8)

axes[2].scatter(runs["train_pr_auc"], runs["pr_auc"], s=60, color=[colors.get(m, "#94a3b8") for m in runs["modelo"]])
low, high = 0, 1.02
axes[2].plot([low, high], [low, high], color="#94a3b8", ls="--")
axes[2].set_xlim(low, high)
axes[2].set_ylim(low, high)
axes[2].set_xlabel("PR-AUC en train")
axes[2].set_ylabel("PR-AUC en test")
axes[2].set_title("Lejos de la diagonal = sobreajuste")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 3. Consultas sobre los runs

`search_runs` acepta filtros con una sintaxis parecida a SQL sobre `params.`, `metrics.` y `tags.`. Por ejemplo: los Random Forest o Gradient Boosting con PR-AUC mayor a 0,5.

In [ ]:
filtered = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.student_id = '{config.student_id}' AND metrics.test_pr_auc > 0.5 AND params.model_type != 'logistic_regression'",
    order_by=["metrics.test_pr_auc DESC"],
)
print(f"{len(filtered)} runs cumplen el filtro")
if len(filtered):
    display(filtered[["tags.mlflow.runName", "params.model_type", "params.feature_set", "metrics.test_pr_auc"]])

## 4. La importancia de las variables del mejor run

Cada run guardó un gráfico de **importancia por permutación**: cuánto empeora el PR-AUC si se desordena al azar cada columna. Lo leemos del artefacto del mejor run.

In [ ]:
best = runs.iloc[0]
print(f"Mejor run: {best.run} ({best.modelo}, {best.features}) · PR-AUC {best.pr_auc}")
try:
    importance = pd.read_csv(mlflow.artifacts.download_artifacts(run_id=best.run_id, artifact_path="importancia_features.csv"))
    display(importance.round(4))
except Exception:
    print("Este run no tiene el artefacto de importancia (los runs a mano del notebook 01 no lo generan).")

## 5. Tabla para tu entrega

Copiá esta tabla en tu `README.md` (son tus 15 mejores runs).

In [ ]:
print(markdown_table(runs, ["run", "modelo", "features", "class_weight", "umbral", "hiperparametros",
                            "train_pr_auc", "pr_auc", "accuracy", "precision", "recall", "f1"]))

## Resumen

- `search_runs` convierte el experimento en una tabla consultable: se filtra y se ordena como cualquier dato.
- La métrica de selección importa: **accuracy** premia no alertar; **PR-AUC** premia ordenar bien el riesgo.
- Comparar `train_pr_auc` con el PR-AUC de test muestra el **sobreajuste**.
- El umbral no cambia el PR-AUC: sólo mueve el punto de trabajo entre precision y recall.